# Delta Lake com Apache Spark

Este notebook demonstra as operações fundamentais do **Delta Lake** com **PySpark**:
- Criação do ambiente
- Carga inicial de dados (INSERT)
- Atualização de registros (UPDATE)
- Remoção de registros (DELETE)
- Histórico e Time Travel

**Dataset:** Vendas de E-commerce

## 1. Configuração do SparkSession com Delta Lake

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col, lit
from delta import *
from delta.tables import DeltaTable

builder = SparkSession.builder \
    .appName("Delta Lake Demo") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "2g")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"✅ SparkSession criada com sucesso!")
print(f"   Versão do Spark : {spark.version}")
print(f"   Master          : {spark.sparkContext.master}")

## 2. Definição do Schema e Dataset

In [ ]:
DELTA_PATH = "./delta-warehouse/vendas"

schema = StructType([
    StructField("id",         IntegerType(), False),
    StructField("produto",    StringType(),  True),
    StructField("categoria",  StringType(),  True),
    StructField("quantidade", IntegerType(), True),
    StructField("preco",      DoubleType(),  True),
    StructField("data_venda", StringType(),  True),
    StructField("vendedor",   StringType(),  True),
    StructField("status",     StringType(),  True),
])

dados_iniciais = [
    (1,  "Notebook Dell",    "Eletrônicos", 2, 3500.00, "2024-01-15", "Ana Silva",    "entregue"),
    (2,  "Camiseta Polo",    "Roupas",      5,   89.90, "2024-01-20", "Bruno Costa",  "entregue"),
    (3,  "Tênis Running",    "Calçados",    1,  450.00, "2024-02-01", "Carlos Lima",  "pago"),
    (4,  "Mouse Gamer",      "Eletrônicos", 3,  250.00, "2024-02-10", "Diana Ramos",  "entregue"),
    (5,  "Calça Jeans",      "Roupas",      2,  199.90, "2024-02-14", "Eduardo Melo", "entregue"),
    (6,  "Smartphone X",     "Eletrônicos", 1, 2800.00, "2024-02-20", "Ana Silva",    "pago"),
    (7,  "Mochila Viagem",   "Acessórios",  1,  350.00, "2024-03-01", "Bruno Costa",  "pendente"),
    (8,  "Teclado Mecânico", "Eletrônicos", 1,  450.00, "2024-03-05", "Carlos Lima",  "pago"),
    (9,  "Vestido Floral",   "Roupas",      3,  159.90, "2024-03-10", "Diana Ramos",  "entregue"),
    (10, "Monitor 27'",      "Eletrônicos", 1, 1800.00, "2024-03-15", "Eduardo Melo", "pendente"),
    (11, "Sandália Couro",   "Calçados",    2,  280.00, "2024-03-18", "Ana Silva",    "entregue"),
    (12, "Câmera DSLR",      "Eletrônicos", 1, 4200.00, "2024-03-20", "Bruno Costa",  "pago"),
    (13, "Jaqueta Denim",    "Roupas",      1,  320.00, "2024-03-22", "Carlos Lima",  "cancelado"),
    (14, "Headset USB",      "Eletrônicos", 2,  180.00, "2024-03-25", "Diana Ramos",  "cancelado"),
    (15, "Relógio Analógico","Acessórios",  1,  520.00, "2024-03-28", "Eduardo Melo", "pendente"),
]

print(f"✅ Schema definido com {len(schema.fields)} campos.")
print(f"   Total de registros iniciais: {len(dados_iniciais)}")

## 3. INSERT — Carga Inicial

Criamos o DataFrame e escrevemos no formato **Delta Lake** com `mode('overwrite')` para criar a tabela.

In [ ]:
df_inicial = spark.createDataFrame(dados_iniciais, schema)

df_inicial.write.format("delta") \
    .mode("overwrite") \
    .save(DELTA_PATH)

print("✅ Tabela Delta criada com sucesso!")
print(f"   Caminho: {DELTA_PATH}\n")

df_leitura = spark.read.format("delta").load(DELTA_PATH)
print(f"   Registros inseridos: {df_leitura.count()}")
df_leitura.show(truncate=False)

### 3.1 INSERT — Append de Novos Registros

Simulamos a chegada de novos pedidos com `mode('append')`.

In [ ]:
novos_pedidos = [
    (16, "Fone Bluetooth",  "Eletrônicos", 1,  299.90, "2024-04-01", "Ana Silva",    "pendente"),
    (17, "Bolsa de Couro",  "Acessórios",  1,  480.00, "2024-04-02", "Bruno Costa",  "pendente"),
    (18, "Tênis Social",    "Calçados",    1,  390.00, "2024-04-03", "Carlos Lima",  "pendente"),
]

df_novos = spark.createDataFrame(novos_pedidos, schema)

df_novos.write.format("delta") \
    .mode("append") \
    .save(DELTA_PATH)

df_atual = spark.read.format("delta").load(DELTA_PATH)
print(f"✅ Novos registros inseridos!")
print(f"   Total de registros agora: {df_atual.count()}")
df_atual.filter(col("id") >= 16).show(truncate=False)

## 4. UPDATE — Atualizando Registros

Usamos a API `DeltaTable` para realizar atualizações. O Delta Lake cria um novo snapshot sem modificar os arquivos antigos.

In [ ]:
deltaTable = DeltaTable.forPath(spark, DELTA_PATH)

print("--- Antes do UPDATE ---")
spark.read.format("delta").load(DELTA_PATH) \
    .filter(col("status") == "pendente") \
    .select("id", "produto", "vendedor", "status") \
    .show(truncate=False)

# UPDATE: todos os pedidos 'pendente' passam para 'pago'
deltaTable.update(
    condition = col("status") == "pendente",
    set       = {"status": lit("pago")}
)

print("--- Após o UPDATE (status pendente → pago) ---")
spark.read.format("delta").load(DELTA_PATH) \
    .filter(col("id").isin([7, 10, 15, 16, 17, 18])) \
    .select("id", "produto", "vendedor", "status") \
    .show(truncate=False)

In [ ]:
# UPDATE 2: pedidos 'pago' com data anterior a 2024-03-01 passam para 'entregue'
deltaTable.update(
    condition = (col("status") == "pago") & (col("data_venda") < "2024-03-01"),
    set       = {"status": lit("entregue")}
)

print("✅ UPDATE 2: pagos antigos → entregue")
spark.read.format("delta").load(DELTA_PATH) \
    .groupBy("status").count() \
    .orderBy("status") \
    .show()

## 5. DELETE — Removendo Registros

Removemos todos os pedidos com status `cancelado`. O Delta Lake cria um novo snapshot marcando os arquivos antigos como removidos.

In [ ]:
print("--- Registros a serem deletados (status = 'cancelado') ---")
spark.read.format("delta").load(DELTA_PATH) \
    .filter(col("status") == "cancelado") \
    .show(truncate=False)

deltaTable.delete(condition = col("status") == "cancelado")

df_pos_delete = spark.read.format("delta").load(DELTA_PATH)
print(f"✅ DELETE executado!")
print(f"   Registros restantes: {df_pos_delete.count()}")
df_pos_delete.groupBy("status").count().orderBy("status").show()

## 6. Histórico de Operações e Time Travel

In [ ]:
# Ver histórico completo da tabela
print("=== Histórico de Operações ===")
deltaTable.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

In [ ]:
# Time Travel: consultar versão 0 (carga inicial — antes de qualquer modificação)
print("=== Time Travel: Versão 0 (estado inicial) ===")
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(DELTA_PATH)

print(f"   Registros na versão 0: {df_v0.count()}")
df_v0.groupBy("status").count().orderBy("status").show()

print("\n=== Comparação: registros cancelados existem na v0 mas não na versão atual ===")
print(f"   Cancelados na v0      : {df_v0.filter(col('status') == 'cancelado').count()}")
print(f"   Cancelados na versão atual: {spark.read.format('delta').load(DELTA_PATH).filter(col('status') == 'cancelado').count()}")

## 7. Leitura Final e Estatísticas

In [ ]:
df_final = spark.read.format("delta").load(DELTA_PATH)

print("=== Estado Final da Tabela ===")
df_final.show(truncate=False)

print("\n=== Estatísticas por Categoria ===")
from pyspark.sql.functions import sum as spark_sum, avg, count

df_final.groupBy("categoria") \
    .agg(
        count("id").alias("total_pedidos"),
        spark_sum("quantidade").alias("total_itens"),
        spark_sum((col("preco") * col("quantidade"))).alias("receita_total"),
        avg("preco").alias("preco_medio")
    ) \
    .orderBy("receita_total", ascending=False) \
    .show(truncate=False)

spark.stop()
print("\n✅ SparkSession encerrada.")